# Wizualizacja Uprawnień Looker w Plotly

Poniższy kod wczytuje plik `looker_graph_all.json` wygenerowany przez skrypt w Pythonie i tworzy interaktywny graf z użyciem `NetworkX` i `Plotly`.

In [ ]:
!pip install networkx plotly

In [ ]:
import json
import networkx as nx
import plotly.graph_objects as go

# 1. Wczytanie danych
file_name = "looker_graph_all.json" # Zmień na swój plik, np. looker_graph_mobile.json
try:
    with open(file_name, "r") as f:
        graph_data = json.load(f)
except FileNotFoundError:
    print(f"Plik {file_name} nie został znaleziony. Upewnij się, że uruchomiłeś permissions_graph_extractor.py")
    graph_data = {"nodes": [], "edges": []}

# 2. Tworzenie grafu skierowanego
G = nx.DiGraph()

# Mapowanie typów węzłów na kolory
type_colors = {
    "model": "#FF0000",
    "explore": "#FF7F00",
    "dashboard": "#0000FF",
    "folder": "#00FF00",
    "model_set": "#8B00FF",
    "role": "#00FFFF",
    "group": "#FF00FF",
    "user": "#8B4513",
    "user_attribute": "#FFC0CB",
    "access_grant": "#FFFF00"
}

# Dodawanie węzłów i krawędzi
for node in graph_data.get("nodes", []):
    G.add_node(node["id"], label=node.get("label", node["id"]), type=node.get("type", "unknown"))

for edge in graph_data.get("edges", []):
    # Jeśli node nie istnieje w grafie, dodajemy go, aby zapobiec błędom powiązań (zabezpieczenie)
    if not G.has_node(edge["source"]):
        G.add_node(edge["source"], label=edge["source"], type="unknown")
    if not G.has_node(edge["target"]):
        G.add_node(edge["target"], label=edge["target"], type="unknown")
        
    G.add_edge(edge["source"], edge["target"], type=edge.get("type", ""))

print(f"Załadowano {G.number_of_nodes()} węzłów oraz {G.number_of_edges()} krawędzi.")

In [ ]:
# 3. Generowanie układu (layout)
# Spring layout działa jak symulacja fizyczna odpychających się węzłów
pos = nx.spring_layout(G, seed=42, k=0.15, iterations=50)

# 4. Przygotowanie ścieżek dla krawędzi w Plotly
edge_x = []
edge_y = []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

# 5. Przygotowanie węzłów w Plotly
node_x = []
node_y = []
node_colors = []
node_texts = []
hover_texts = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    
    n_type = G.nodes[node].get("type", "unknown")
    node_colors.append(type_colors.get(n_type, "#cccccc"))
    
    label = G.nodes[node].get('label', str(node))
    # Długie nazwy ucinamy by graf był czytelny
    short_label = label[:15] + "..." if len(label) > 15 else label
    node_texts.append(short_label)
    hover_texts.append(f"Typ: {n_type}<br>Nazwa: {label}")

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    textposition="bottom center",
    text=node_texts,
    hoverinfo='text',
    hovertext=hover_texts,
    marker=dict(
        showscale=False,
        color=node_colors,
        size=12,
        line_width=1,
        line_color="white"
    )
)

# 6. Wyrysowanie grafu
fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                title='Struktura i Uprawnienia - Looker Graph',
                title_font_size=18,
                showlegend=False,
                hovermode='closest',
                margin=dict(b=20,l=5,r=5,t=40),
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                width=1000,
                height=800
             )
)

fig.show()